In [1]:
# ============================================================
# This step installs the Python packages required for:
# 1. Protein language model embedding generation
# 2. Deep learning model development
# 3. Data processing and evaluation
#
# NOTE:
# Users only need to run this step once per fresh Colab session.
# ============================================================

!pip install fair-esm
!pip install torch
!pip install tensorflow
!pip install scikit-learn
!pip install h5py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 10.9 MB/s eta 0:00:00


In [2]:
# ============================================================
# This step loads the libraries required for:
# 1. Numerical computation and data handling
# 2. Sequence embedding generation using ESM2
# 3. CNN model training and evaluation
#
# NOTE:
# If a GPU is available in Google Colab, it will be detected and used.
# ============================================================

import gc
import math
import esm
import pandas as pd
import numpy as np
import tensorflow as tf
import torch

from keras.layers import Input, Dense, Activation, BatchNormalization, Flatten, Conv1D
from keras.layers import Dropout, MaxPooling1D
from keras.models import Model, load_model
from keras.optimizers import SGD
from keras.callbacks import ModelCheckpoint, LearningRateScheduler, EarlyStopping

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, roc_auc_score


# Check GPU availability
if tf.test.gpu_device_name():
    print('GPU found')
    tf.config.experimental.set_visible_devices(tf.config.list_physical_devices('GPU')[0], 'GPU') # set the deep learning with GPU
else:
    print("No GPU found")


GPU found


In [3]:
# ============================================================
# This function converts protein sequences into 320-dimensional
# numerical embeddings using the pretrained ESM2 model.
#
# NOTE:
# This implementation is suitable for small datasets and protocol testing.
# For larger datasets, users may process sequences in batches to reduce
# memory usage and improve computational efficiency.
# ============================================================


def esm_embeddings_320(esm2, esm2_alphabet, peptide_sequence_list):
  import collections

  if torch.cuda.is_available():
    device = torch.device("cuda")
  else:
    device = torch.device("cpu")

  esm2 = esm2.eval().to(device)
  batch_converter = esm2_alphabet.get_batch_converter()

  # Convert sequence to model tokens
  batch_labels, batch_strs, batch_tokens = batch_converter(peptide_sequence_list)
  batch_lens = (batch_tokens != esm2_alphabet.padding_idx).sum(1)
  batch_tokens = batch_tokens.to(device)

  # Extract final-layer sequence representations
  with torch.no_grad():
      results = esm2(batch_tokens, repr_layers=[6], return_contacts=False)

  token_representations = results["representations"][6].cpu()

  # # Average residue-level embeddings to obtain one vector per sequence
  # NOTE: token 0 is always a beginning-of-sequence token, so the first residue is token 1.
  sequence_representations = []
  for i, tokens_len in enumerate(batch_lens):
      sequence_representations.append(token_representations[i, 1 : tokens_len - 1].mean(0))

  # Convert embeddings into DataFrame
  embeddings_results = collections.defaultdict(list)
  for i in range(len(sequence_representations)):
      each_seq_rep = sequence_representations[i].tolist()
      for each_element in each_seq_rep:
          embeddings_results[i].append(each_element)

  embeddings_results = pd.DataFrame(embeddings_results).T

  del  batch_labels, batch_strs, batch_tokens, results, token_representations
  torch.cuda.empty_cache()
  gc.collect()

  return embeddings_results


In [6]:
# ============================================================
# This step:
# 1. Loads the pretrained ESM2 model
# 2. Reads the example sequence dataset
# 3. Generates embeddings for each sequence
# 4. Saves the embedding matrix for downstream model development
#
# NOTE:
# The short example dataset is intentionally small and is provided only
# for quick testing of the workflow.
# ============================================================

# Load ESM2 model
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()

# Load dataset
dataset = pd.read_excel('pLM4Alg_protocol_example_allergens_short_dataset.xlsx', na_filter = False)

# Generate embeddings
sequence_list = dataset['sequence']
embeddings_results = pd.DataFrame()

for seq in sequence_list:
    tuple_sequence = tuple([seq, seq])
    peptide_sequence_list = []
    peptide_sequence_list.append(tuple_sequence) # build a summarize list variable including all the sequence information

    one_seq_embeddings = esm_embeddings_320(model, alphabet, peptide_sequence_list)
    embeddings_results= pd.concat([embeddings_results,one_seq_embeddings])

# Save embeddings
embedding_file = 'whole_sample_dataset_esm2_t6_8M_UR50D_unified_320_dimension.csv'
embeddings_results.to_csv(embedding_file, index=False)

print(f"Saved embeddings to: {embedding_file}")


Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t6_8M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t6_8M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t6_8M_UR50D-contact-regression.pt
Saved embeddings to: whole_sample_dataset_esm2_t6_8M_UR50D_unified_320_dimension.csv


In [7]:
# ============================================================
# This step:
# 1. Loads the labels from the example dataset
# 2. Loads the precomputed embeddings
# 3. Verifies that labels and embeddings have matching sample counts
# 4. Splits the dataset into training and independent test sets
#
# NOTE:
# Users should ensure that the embedding file was generated from the
# same dataset used to load the labels.
# ============================================================

# Load labels
dataset = pd.read_excel('pLM4Alg_protocol_example_allergens_short_dataset.xlsx', na_filter=False)
y = dataset['label'].to_numpy()

# Load embeddings
X_data = pd.read_csv('whole_sample_dataset_esm2_t6_8M_UR50D_unified_320_dimension.csv', header=0)
X = X_data.to_numpy()

print("Number of labels:", len(y))
print("Embedding matrix shape:", X.shape)

# Ensure labels and embeddings match
if len(X) != len(y):
    raise ValueError(
        f"Mismatch between embeddings ({len(X)}) and labels ({len(y)}). "
        "Please ensure that the embeddings were generated from the same dataset."
    )

# Initial train-test split
random_num = 1
X_train_whole, X_test, y_train_whole, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_num, shuffle=True, stratify=y
)

# **Further split training pool into training and validation sets**
X_train, X_val, y_train, y_val = train_test_split(
    X_train_whole, y_train_whole, test_size=0.2, random_state=random_num, stratify=y_train_whole
)

print("Training set shape:", X_train.shape)
print("Validation set shape:", X_val.shape)
print("Independent test set shape:", X_test.shape)


Number of labels: 22
Embedding matrix shape: (22, 320)
Training set shape: (13, 320)
Validation set shape: (4, 320)
Independent test set shape: (5, 320)


In [8]:
# Normalize the X data range
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

scaler.fit(X_train)
X_train = scaler.transform(X_train) # normalize X to 0-1 range
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [9]:
# ============================================================
# This function trains one CNN model using:
# 1. A training subset
# 2. A validation subset
# 3. Selected hyperparameter values
#
# NOTE:
# Validation-based checkpointing and early stopping are included here
# because this function is intended for hyperparameter evaluation.
# ============================================================

def train_model_cv(X_train, X_valid, y_train, y_valid, filter_size, kernel_size, stride_size, node_units):

    input_shape = (320, 1)
    inputs = Input(input_shape)

    x = Conv1D(filter_size, kernel_size, strides=stride_size, name='layer_conv2', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling1D(2, name='MaxPool2', padding="same")(x)
    x = Dropout(0.15)(x)
    x = Flatten()(x)
    x = Dense(node_units, activation='relu', name='fc1')(x)
    x = Dropout(0.15)(x)
    outputs = Dense(2, activation='softmax', name='fc2')(x)

    model = Model(inputs=inputs, outputs=outputs, name='Predict')

    # Define optimizer
    momentum = 0.5
    sgd = SGD(learning_rate=0.01, momentum=momentum, nesterov=False)

    # Compile model
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=sgd,
        metrics=['accuracy']
    )

    # Learning rate decay
    def step_decay(epoch):
        initial_lrate = 0.1
        drop = 0.6
        epochs_drop = 3.0
        return initial_lrate * math.pow(drop, math.floor((1 + epoch) / epochs_drop))

    lr = LearningRateScheduler(step_decay)

    # Early stopping
    early_stop = EarlyStopping(
        monitor='val_accuracy',
        patience=20,
        verbose=0,
        restore_best_weights=True
    )

    # Save best model
    checkpoint_file = 'best_model_grid_320.keras'
    mc = ModelCheckpoint(
        checkpoint_file,
        monitor='val_accuracy',
        mode='max',
        verbose=0,
        save_best_only=True,
        save_weights_only=False
    )

    callbacks_list = [lr, early_stop, mc]

    model.fit(
        X_train, y_train,
        validation_data=(X_valid, y_valid),
        epochs=200,
        callbacks=callbacks_list,
        batch_size=32,
        verbose=0
    )

    saved_model = load_model(checkpoint_file)
    return saved_model


In [10]:
# ============================================================
# Performance evaluation function
# ============================================================

def check_performance(saved_model, X_eval, y_eval):
    predicted_probability = saved_model.predict(X_eval, batch_size=1, verbose=0)
    predicted_class = np.argmax(predicted_probability, axis=1)

    # Force binary confusion matrix order
    TN, FP, FN, TP = confusion_matrix(y_eval, predicted_class, labels=[0, 1]).ravel()

    ACC = (TP + TN) / (TP + TN + FP + FN)

    Sn = TP / (TP + FN) if (TP + FN) != 0 else np.nan
    Sp = TN / (TN + FP) if (TN + FP) != 0 else np.nan

    denom = math.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    MCC = ((TP * TN - FP * FN) / denom) if denom != 0 else np.nan

    BACC = np.nanmean([Sn, Sp])

    try:
        AUC = roc_auc_score(y_eval, predicted_probability[:, 1])
    except ValueError:
        AUC = np.nan

    return ACC, Sn, Sp, MCC, BACC, AUC


In [18]:
# ============================================================
# Code Snippet 9: Cross-validation evaluation of selected models
# This step:
# 1. Performs stratified k-fold cross-validation
# 2. Splits data into training and validation folds
# 3. Applies normalization within each fold
# 4. Trains CNN models using grid search over selected hyperparameters
# 5. Evaluates ACC, Sn, Sp, MCC, BACC, and AUC across folds
# 6. Reports mean ± standard deviation for each model
# 7. Reports overall mean ± standard deviation across all folds and models
# 8. Selects the best hyperparameter set based on MCC
# ============================================================

# Reduced grid search space for quick protocol testing
CNN_channel = [16, 32]
dense_node = [32, 128]
kernel_size = [3, 6]
stride_size = [1, 2]

print("Total number of hyperparameter combinations:",
      len(CNN_channel) * len(dense_node) * len(kernel_size) * len(stride_size))

kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)

# Per-model summary storage
model_summaries = []

# Mean metric storage for best hyperparameter selection
mean_MCC_values = []

# Global fold-level storage across all models
global_ACC = []
global_Sn = []
global_Sp = []
global_MCC = []
global_BACC = []
global_AUC = []

model_id = 0

for f in CNN_channel:
    for d in dense_node:
        for k in kernel_size:
            for s in stride_size:

                ACC_list = []
                Sn_list = []
                Sp_list = []
                MCC_list = []
                BACC_list = []
                AUC_list = []

                for train_ix, val_ix in kfold.split(X_train_whole, y_train_whole):

                    X_train_fold = X_train_whole[train_ix]
                    X_val_fold   = X_train_whole[val_ix]
                    y_train_fold = y_train_whole[train_ix]
                    y_val_fold   = y_train_whole[val_ix]

                    # Normalize within each fold to avoid data leakage
                    scaler = MinMaxScaler()
                    scaler.fit(X_train_fold)

                    X_train_fold = scaler.transform(X_train_fold).reshape((-1, 320, 1))
                    X_val_fold   = scaler.transform(X_val_fold).reshape((-1, 320, 1))

                    # Train one model for current parameter combination
                    model = train_model_cv(
                        X_train_fold,
                        X_val_fold,
                        y_train_fold,
                        y_val_fold,
                        f,   # filter size
                        k,   # kernel size
                        s,   # stride size
                        d    # dense units
                    )

                    # Evaluate current fold
                    ACC, Sn, Sp, MCC, BACC, AUC = check_performance(model, X_val_fold, y_val_fold)

                    # Store per-model fold metrics
                    ACC_list.append(ACC)
                    Sn_list.append(Sn)
                    Sp_list.append(Sp)
                    MCC_list.append(MCC)
                    BACC_list.append(BACC)
                    AUC_list.append(AUC)

                    # Store global metrics
                    global_ACC.append(ACC)
                    global_Sn.append(Sn)
                    global_Sp.append(Sp)
                    global_MCC.append(MCC)
                    global_BACC.append(BACC)
                    global_AUC.append(AUC)

                    del model
                    gc.collect()

                # Compute per-model mean ± SD
                acc_mean  = np.nanmean(np.array(ACC_list, dtype=float))
                acc_std   = np.nanstd(np.array(ACC_list, dtype=float), ddof=1)

                sn_mean   = np.nanmean(np.array(Sn_list, dtype=float))
                sn_std    = np.nanstd(np.array(Sn_list, dtype=float), ddof=1)

                sp_mean   = np.nanmean(np.array(Sp_list, dtype=float))
                sp_std    = np.nanstd(np.array(Sp_list, dtype=float), ddof=1)

                mcc_mean  = np.nanmean(np.array(MCC_list, dtype=float))
                mcc_std   = np.nanstd(np.array(MCC_list, dtype=float), ddof=1)

                bacc_mean = np.nanmean(np.array(BACC_list, dtype=float))
                bacc_std  = np.nanstd(np.array(BACC_list, dtype=float), ddof=1)

                auc_mean  = np.nanmean(np.array(AUC_list, dtype=float))
                auc_std   = np.nanstd(np.array(AUC_list, dtype=float), ddof=1)

                model_summaries.append({
                    "model_id": model_id,
                    "filter_size": f,
                    "dense_units": d,
                    "kernel_size": k,
                    "stride_size": s,
                    "ACC": f"{acc_mean:.3f}±{acc_std:.3f}",
                    "Sn": f"{sn_mean:.3f}±{sn_std:.3f}",
                    "Sp": f"{sp_mean:.3f}±{sp_std:.3f}",
                    "MCC": f"{mcc_mean:.3f}±{mcc_std:.3f}",
                    "BACC": f"{bacc_mean:.3f}±{bacc_std:.3f}",
                    "AUC": f"{auc_mean:.3f}±{auc_std:.3f}"
                })

                mean_MCC_values.append(mcc_mean)

                print(f"Model {model_id}")
                print(f"Parameters: filters={f}, dense_units={d}, kernel_size={k}, stride_size={s}")
                print(f"ACC : {acc_mean:.3f}±{acc_std:.3f}")
                print(f"Sn  : {sn_mean:.3f}±{sn_std:.3f}")
                print(f"Sp  : {sp_mean:.3f}±{sp_std:.3f}")
                print(f"MCC : {mcc_mean:.3f}±{mcc_std:.3f}")
                print(f"BACC: {bacc_mean:.3f}±{bacc_std:.3f}")
                print(f"AUC : {auc_mean:.3f}±{auc_std:.3f}")
                print("-" * 60)

                model_id += 1

# Overall CV performance across all folds and all tested models
print("\nOverall Cross-validation Performance:")
print(f"ACC : {np.nanmean(global_ACC):.3f}±{np.nanstd(global_ACC, ddof=1):.3f}")
print(f"Sn  : {np.nanmean(global_Sn):.3f}±{np.nanstd(global_Sn, ddof=1):.3f}")
print(f"Sp  : {np.nanmean(global_Sp):.3f}±{np.nanstd(global_Sp, ddof=1):.3f}")
print(f"MCC : {np.nanmean(global_MCC):.3f}±{np.nanstd(global_MCC, ddof=1):.3f}")
print(f"BACC: {np.nanmean(global_BACC):.3f}±{np.nanstd(global_BACC, ddof=1):.3f}")
print(f"AUC : {np.nanmean(global_AUC):.3f}±{np.nanstd(global_AUC, ddof=1):.3f}")

# Select best hyperparameters based on MCC
best_index = int(np.nanargmax(mean_MCC_values))
best_params = {
    "filter_size": model_summaries[best_index]["filter_size"],
    "dense_units": model_summaries[best_index]["dense_units"],
    "kernel_size": model_summaries[best_index]["kernel_size"],
    "stride_size": model_summaries[best_index]["stride_size"]
}

print("\nBest hyperparameter set selected based on cross-validation MCC:")
print(best_params)

Total number of hyperparameter combinations: 16
Model 0
Parameters: filters=16, dense_units=32, kernel_size=3, stride_size=1
ACC : 0.833±0.289
Sn  : 0.667±0.577
Sp  : 1.000±0.000
MCC : 1.000±0.000
BACC: 0.833±0.289
AUC : 1.000±0.000
------------------------------------------------------------
Model 1
Parameters: filters=16, dense_units=32, kernel_size=3, stride_size=2
ACC : 1.000±0.000
Sn  : 1.000±0.000
Sp  : 1.000±0.000
MCC : 1.000±0.000
BACC: 1.000±0.000
AUC : 1.000±0.000
------------------------------------------------------------
Model 2
Parameters: filters=16, dense_units=32, kernel_size=6, stride_size=1
ACC : 1.000±0.000
Sn  : 1.000±0.000
Sp  : 1.000±0.000
MCC : 1.000±0.000
BACC: 1.000±0.000
AUC : 1.000±0.000
------------------------------------------------------------
Model 3
Parameters: filters=16, dense_units=32, kernel_size=6, stride_size=2
ACC : 1.000±0.000
Sn  : 1.000±0.000
Sp  : 1.000±0.000
MCC : 1.000±0.000
BACC: 1.000±0.000
AUC : 1.000±0.000
-----------------------------

In [20]:
# ============================================================
# Code Snippet 10: Final model training
# This step:
# 1. Uses the best hyperparameters selected from cross-validation
# 2. Trains the CNN model on the full training dataset
# 3. Applies consistent preprocessing (normalization + reshaping)
# 4. Evaluates final model performance on the independent test dataset
# ============================================================

def train_model_final(X_train, y_train, filter_size, kernel_size, stride_size, node_units):

    input_shape = (320, 1)
    inputs = Input(input_shape)

    x = Conv1D(filter_size, kernel_size, strides=stride_size, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling1D(2, padding="same")(x)
    x = Dropout(0.15)(x)
    x = Flatten()(x)
    x = Dense(node_units, activation='relu')(x)
    x = Dropout(0.15)(x)
    outputs = Dense(2, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)

    # Optimizer
    sgd = SGD(learning_rate=0.01, momentum=0.5, nesterov=False)

    # Compile model
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=sgd,
        metrics=['accuracy']
    )

    # Learning rate scheduler
    def step_decay(epoch):
        initial_lrate = 0.1
        drop = 0.6
        epochs_drop = 3.0
        return initial_lrate * math.pow(drop, math.floor((1 + epoch) / epochs_drop))

    lr = LearningRateScheduler(step_decay)

    # Train final model
    model.fit(
        X_train,
        y_train,
        epochs=45,
        callbacks=[lr],
        batch_size=32,
        verbose=0
    )

    return model


# Normalize and reshape full training and independent test datasets

scaler = MinMaxScaler()
scaler.fit(X_train_whole)

X_train_final = scaler.transform(X_train_whole).reshape((-1, 320, 1))
X_test_final  = scaler.transform(X_test).reshape((-1, 320, 1))


# Train final model using the selected best hyperparameters

final_model = train_model_final(
    X_train_final,
    y_train_whole,
    filter_size=best_params["filter_size"],
    kernel_size=best_params["kernel_size"],
    stride_size=best_params["stride_size"],
    node_units=best_params["dense_units"]
)


# Evaluate final model on the independent test dataset
ACC, Sn, Sp, MCC, BACC, AUC = check_performance(final_model, X_test_final, y_test)

print("\nBest hyperparameter set used for final model:")
print(best_params)

print("\nFinal Test Performance:")
print("ACC :", ACC)
print("Sn  :", Sn)
print("Sp  :", Sp)
print("MCC :", MCC)
print("BACC:", BACC)
print("AUC :", AUC)


Best hyperparameter set used for final model:
{'filter_size': 16, 'dense_units': 32, 'kernel_size': 3, 'stride_size': 1}

Final Test Performance:
ACC : 1.0
Sn  : 1.0
Sp  : 1.0
MCC : 1.0
BACC: 1.0
AUC : 1.0


**Model Application for New Dataset**



In [30]:
#Generate embeddings for a new dataset.
# load pretrained ESM2 model
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()

# load new dataset
dataset = pd.read_excel('new_dataset.xlsx', header=0)

# generate embeddings
sequence_list = dataset['sequence']
embeddings_results = pd.DataFrame()

for seq in sequence_list:
    tuple_sequence = tuple([seq, seq])
    peptide_sequence_list = [tuple_sequence]

    one_seq_embeddings = esm_embeddings_320(model, alphabet, peptide_sequence_list)

    embeddings_results = pd.concat([embeddings_results, one_seq_embeddings])

# save embeddings
embeddings_results.to_csv('new_data_embeddings_320.csv')


In [31]:
#Normalize new dataset
# load new embeddings
X_new_embeddings = pd.read_csv('new_data_embeddings_320.csv', header=0, index_col=0)
X_new = np.array(X_new_embeddings)

# apply previously fitted scaler
X_new = scaler.transform(X_new)

# reshape for CNN input
X_new = X_new.reshape((X_new.shape[0], 320, 1))


In [32]:
#Check saved files
import os
print(os.listdir())

['.config', 'new_data_embeddings_320.csv', 'new_dataset.xlsx', 'best_model_grid_320.keras', 'pLM4Alg_protocol_example_allergens_short_dataset.xlsx', 'new_data_prediction_result.xlsx', 'whole_sample_dataset_esm2_t6_8M_UR50D_unified_320_dimension.csv', '.ipynb_checkpoints', 'sample_data']


In [33]:
#Prediction on new dataset with probability
# load trained model
saved_model = load_model('best_model_grid_320.keras')

# generate predictions
predicted_probability = saved_model.predict(X_new, batch_size=1)

predicted_class = []
prediction_confidence = []

for i in range(predicted_probability.shape[0]):
    probs = predicted_probability[i]
    index = np.argmax(probs)

    predicted_class.append(index)
    prediction_confidence.append(np.max(probs))  # confidence score

predicted_class = np.array(predicted_class)
prediction_confidence = np.array(prediction_confidence)

# store results
dataset['predicted_label'] = predicted_class
dataset['confidence_score'] = prediction_confidence

# save results
dataset.to_excel('new_data_prediction_result.xlsx', index=False)

# display results
print(dataset[['sequence', 'predicted_label', 'confidence_score']].head())


6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step  
                                 sequence  predicted_label  confidence_score
0                            VIACVGETLEQR                1          0.982257
1               VICDHLGLGVKTGLPYIWHSKASNP                1          0.981523
2                       VIPGFMCQGGDFTAGNG                1          0.984346
3  MKVRASVKKLCRNCKIIRRDGIVRVICSAEPRHKQRQG                1          0.980769
4   MKVNPSVKPICDKCRVIRRHGRVMVICQDPRHKQRQG                1          0.983678
